# Visualisation des études — Guidance, XGR et Outlines

Notebook principal pour consulter **l'EDA, les features raffinées, les modèles retenus, leurs scores et matrices de confusion, les arbres explicatifs et leurs règles, puis Kubernetes**.

Les résultats enregistrés sont consultables sans réentraîner de modèle. Les cellules de prédiction peuvent également **réexécuter les modèles retenus**, sur leur partition de test et sur Kubernetes. Les recalculs sont affichés séparément des scores historiques.

Les modèles prédictifs principaux et les arbres explicatifs sont deux objets distincts. `DATASET_CHOICE` pilote l'EDA et les analyses raffinées ; il ne change pas la partition des modèles GitHub. Les modèles entraînés sur Kubernetes constituent une expérience séparée de l'évaluation externe.

Voir les guides [results](results/README.md), [coverage_prediction](coverage_prediction/README.md) et [scripts](scripts/README.md).


## 1. Paramètres

Choisissez un ou plusieurs frameworks. `global` correspond à `Github_global` pour Guidance/XGR et `Github_global_no_hard` pour Outlines. `Kubernetes` est également accepté, ainsi que les datasets GitHub individuels.

Laissez les options de régénération désactivées pour consulter les sorties existantes. Si activées, elles écrivent seulement dans `notebook_outputs/`. L'extraction V1 n'est pas relancée : son module est actuellement absent. Les tables V1 disponibles restent consultables et utilisables pour recalculer la V2.


In [ ]:
from pathlib import Path
import csv
import html
import json
import subprocess
import sys
from IPython.display import HTML, Markdown, display

DATASET_CHOICE = "global"
FRAMEWORKS = ["guidance", "xgr", "outlines"]
TARGETS = ["under", "over"]
EXTERNAL_DATASET = "Kubernetes"
REGENERATE_EDA = False
REGENERATE_REFINED_V2 = False
RUN_MODEL_PREDICTIONS = True
RUN_EXTERNAL_PREDICTIONS = True
MAX_TABLE_ROWS = 12
# Facultatif : Python disposant des versions compatibles avec les pickles (scikit-learn 1.9.0, LightGBM).
MODEL_PYTHON = None


In [ ]:
def find_extension_dir():
    for base in [Path.cwd().resolve(), *Path.cwd().resolve().parents]:
        for candidate in (base, base / "extension_jsonschemabench", base / "jsonschemabench" / "extension_jsonschemabench"):
            if (candidate / "coverage_prediction").is_dir() and (candidate / "scripts").is_dir():
                return candidate
    raise FileNotFoundError("Lancez Jupyter depuis le projet ou le dossier de l'extension.")

EXTENSION_DIR = find_extension_dir()
REPO_ROOT = EXTENSION_DIR.parent
RESULTS_ROOT = EXTENSION_DIR / "results" / "per_dataset_runs"
COVERAGE_ROOT = EXTENSION_DIR / "coverage_prediction"
MODELS_ROOT = COVERAGE_ROOT / "modeles_predictifs"
SCRIPTS_DIR = EXTENSION_DIR / "scripts"
DATA_ROOT = REPO_ROOT / "maskbench" / "data"
OUTPUT_ROOT = EXTENSION_DIR / "notebook_outputs"
V2_OUTPUT_ROOT = OUTPUT_ROOT / "refined_feature_analysis_v2"
venv_python = REPO_ROOT / ".venv" / "bin" / "python"
MODEL_PYTHON = str(MODEL_PYTHON or (venv_python if venv_python.exists() else sys.executable))
GLOBAL_DATASET_BY_FRAMEWORK = {"guidance": "Github_global", "xgr": "Github_global", "outlines": "Github_global_no_hard"}
assert FRAMEWORKS and set(FRAMEWORKS) <= set(GLOBAL_DATASET_BY_FRAMEWORK), "Framework inconnu"
assert TARGETS and set(TARGETS) <= {"under", "over"}, "Cible inconnue"

def resolve_dataset(framework, dataset_choice=None):
    choice = DATASET_CHOICE if dataset_choice is None else dataset_choice
    return GLOBAL_DATASET_BY_FRAMEWORK[framework] if choice.lower() == "global" else choice

def run_dir(framework, dataset_choice=None):
    return RESULTS_ROOT / framework / resolve_dataset(framework, dataset_choice)

print("Extension :", EXTENSION_DIR)
print("Python des modèles :", MODEL_PYTHON)


## 2. Fonctions de lecture et de présentation


In [ ]:
def read_rows(path):
    if not path.is_file() or not path.stat().st_size:
        return []
    with path.open(encoding="utf-8-sig", newline="") as handle:
        return list(csv.DictReader(handle))

def show_rows(rows, title=None, limit=MAX_TABLE_ROWS):
    if title:
        display(Markdown(f"**{title}**"))
    if not rows:
        display(Markdown("Aucune ligne disponible."))
        return
    headers = list(dict.fromkeys(k for row in rows for k in row))
    shown = rows if limit is None else rows[:limit]
    table = '<div style="overflow:auto;max-height:550px"><table><thead><tr>'
    table += ''.join(f'<th>{html.escape(str(k))}</th>' for k in headers) + '</tr></thead><tbody>'
    for row in shown:
        table += '<tr>' + ''.join(f'<td>{html.escape(str(row.get(k, "")))}</td>' for k in headers) + '</tr>'
    table += '</tbody></table></div>'
    display(HTML(table))
    if len(shown) < len(rows):
        display(Markdown(f"Aperçu : {len(shown)} / {len(rows)} lignes."))

def show_csv(path, limit=MAX_TABLE_ROWS):
    if not path.is_file():
        display(Markdown(f"Fichier absent : `{path.relative_to(EXTENSION_DIR)}`"))
        return
    show_rows(read_rows(path), path.name, limit)

def show_report(path):
    if path.is_file():
        display(Markdown(path.read_text(encoding="utf-8")))
    else:
        display(Markdown(f"Rapport absent : `{path.relative_to(EXTENSION_DIR)}`"))

def show_gallery(directory, recursive=True):
    files = sorted(directory.rglob("*.svg") if recursive else directory.glob("*.svg"))
    if not files:
        display(Markdown(f"Aucun SVG : `{directory.relative_to(EXTENSION_DIR)}`"))
        return
    display(Markdown(f"{len(files)} graphiques disponibles — cliquez sur les titres pour les ouvrir."))
    for path in files:
        svg = path.read_text(encoding="utf-8")
        start = svg.find("<svg")
        if start >= 0:
            display(HTML('<details><summary>'+html.escape(str(path.relative_to(directory)))+'</summary>'+svg[start:]+'</details>'))

def show_confusions(rows, title):
    for row in rows:
        if not all(str(row.get(k, "")).strip() for k in ("tn", "fp", "fn", "tp")):
            continue
        label = f"{title} — {row.get('model', '')} {row.get('split', '')}"
        show_rows([
            {"Vérité / Prédiction": "Négatif (pas d'erreur)", "Négatif": row["tn"], "Positif": row["fp"]},
            {"Vérité / Prédiction": "Positif (UNDER ou OVER)", "Négatif": row["fn"], "Positif": row["tp"]},
        ], label)

# Inventaire du périmètre choisi ; l'absence de plots ne signifie pas absence de run.
show_rows([{
    "framework": fw, "dataset EDA": resolve_dataset(fw),
    "résultats par test": (run_dir(fw) / "per_test_results.jsonl").exists(),
    "plots EDA": (run_dir(fw) / "plots").exists(),
    "tables raffinées": (run_dir(fw) / "refined_feature_analysis" / "refined_test_features.csv").exists(),
    "modèles": (MODELS_ROOT / fw).exists(),
} for fw in FRAMEWORKS])


## 3. EDA et profilage, par framework

Affichage des statistiques et de **tous les graphiques disponibles**, y compris les timeouts, erreurs de compilation, UNDER/OVER, tailles et temps. La régénération optionnelle produit une nouvelle analyse dans `notebook_outputs/eda/`.


In [ ]:
def show_eda(framework):
    display(Markdown(f"### {framework} — {resolve_dataset(framework)}"))
    source = run_dir(framework)
    plots = source / "plots"
    if REGENERATE_EDA:
        if not (source / "per_test_results.jsonl").is_file() or not (source / "timing_profile.csv").is_file():
            display(Markdown("Régénération ignorée : résultats par test ou profilage absents."))
        else:
            generated = OUTPUT_ROOT / "eda" / framework / resolve_dataset(framework)
            cmd = [MODEL_PYTHON, str(SCRIPTS_DIR / "analyze_dataset_statistics.py"),
                   "--framework", framework, "--dataset", resolve_dataset(framework),
                   "--results-root", str(RESULTS_ROOT), "--data-root", str(DATA_ROOT), "--output-dir", str(generated)]
            result = subprocess.run(cmd, cwd=REPO_ROOT, capture_output=True, text=True)
            if result.returncode:
                display(Markdown("Échec de la régénération ; les résultats enregistrés restent consultables."))
                print(result.stderr[-3000:])
            else:
                plots = generated
    show_csv(plots / "schema_level_stats.csv")
    show_gallery(plots, recursive=False)

for framework in FRAMEWORKS:
    show_eda(framework)


## 4. Features extraites et analyses raffinées V1/V2

Les aperçus des tables montrent les features réellement disponibles. Les galeries couvrent les familles numériques, objets/regex, combinateurs, `not` et les synthèses. Les graphiques existants sont lus dans les deux organisations de dossiers utilisées par les runs.

`REGENERATE_REFINED_V2=True` recalcule la V2 depuis les tables V1, sans refaire l'extraction initiale. Une absence de données est signalée, sans bloquer les autres frameworks.


In [ ]:
UNDER_FIELDNAMES = [
    "context_family",
    "context_feature",
    "context_value",
    "support_invalid_tests",
    "support_schemas",
    "under_count",
    "under_rate_invalid_only",
    "baseline_under_rate_invalid_only",
    "under_lift_invalid_only",
    "low_support",
]

OVER_FIELDNAMES = [
    "context_family",
    "context_feature",
    "context_value",
    "support_valid_tests",
    "support_schemas",
    "over_count",
    "over_rate_valid_only",
    "baseline_over_rate_valid_only",
    "over_lift_valid_only",
    "low_support",
]


def generate_refined_v2(framework: str, dataset_choice: str | None = None) -> dict[str, Path | int]:
    """Génère les tables et les plots raffinés v2 pour un framework.

    Cette fonction appelle directement les fonctions du script v2 :
    conditional_under_risk, conditional_over_risk, schema_level_risk,
    write_plots, cooccurrence_heatmap, top_examples et write_report.
    """
    selected_dataset = resolve_dataset(framework, dataset_choice)
    source_dir = run_dir(framework, dataset_choice) / "refined_feature_analysis"
    output_data_dir = V2_OUTPUT_ROOT / framework / selected_dataset / "tables"
    output_plot_dir = V2_OUTPUT_ROOT / framework / selected_dataset / "plots"

    test_features = source_dir / "refined_test_features.csv"
    schema_features = source_dir / "refined_schema_features.csv"
    if not test_features.exists() or not schema_features.exists():
        raise FileNotFoundError(
            "Tables raffinées v1 absentes pour "
            f"{framework}/{selected_dataset}. Attendu : {source_dir}"
        )

    output_data_dir.mkdir(parents=True, exist_ok=True)
    output_plot_dir.mkdir(parents=True, exist_ok=True)

    test_rows = refined_v2.read_csv(test_features)
    schema_rows = refined_v2.read_csv(schema_features)

    under_rows = refined_v2.conditional_under_risk(test_rows)
    over_rows = refined_v2.conditional_over_risk(test_rows)
    schema_risk_rows = refined_v2.schema_level_risk(schema_rows, test_rows)

    refined_v2.write_csv(output_data_dir / "under_invalid_only_risk.csv", under_rows, UNDER_FIELDNAMES)
    refined_v2.write_csv(output_data_dir / "over_valid_only_risk.csv", over_rows, OVER_FIELDNAMES)
    refined_v2.write_csv(output_data_dir / "schema_level_context_risk.csv", schema_risk_rows)

    # Ici on génère les plots v2 depuis les tables, au lieu de lire des plots déjà existants.
    refined_v2.write_plots(output_plot_dir, test_rows, under_rows, over_rows)
    refined_v2.cooccurrence_heatmap(
        output_plot_dir / "summary" / "cooccurrence_under_schemas.svg",
        "Keyword co-occurrence in UNDER schemas",
        refined_v2.cooccurrence_rows(schema_rows, DATA_ROOT, "under"),
    )
    refined_v2.cooccurrence_heatmap(
        output_plot_dir / "summary" / "cooccurrence_over_schemas.svg",
        "Keyword co-occurrence in OVER schemas",
        refined_v2.cooccurrence_rows(schema_rows, DATA_ROOT, "over"),
    )

    examples = refined_v2.top_examples(test_rows, under_rows, over_rows, DATA_ROOT)
    refined_v2.write_csv(output_data_dir / "top_context_examples.csv", examples)
    refined_v2.write_report(output_data_dir / "refined_feature_analysis_v2_report.md", test_rows, under_rows, over_rows, schema_risk_rows)

    return {
        "framework": framework,
        "dataset": selected_dataset,
        "source_dir": source_dir,
        "output_data_dir": output_data_dir,
        "output_plot_dir": output_plot_dir,
        "n_test_rows": len(test_rows),
        "n_schema_rows": len(schema_rows),
    }


In [ ]:
def show_refined(framework):
    display(Markdown(f"### Features — {framework} / {resolve_dataset(framework)}"))
    source = run_dir(framework)
    for name in ("refined_schema_features.csv", "refined_test_features.csv"):
        show_csv(source / "refined_feature_analysis" / name, limit=3)
    for version in ("refined_feature_analysis", "refined_feature_analysis_v2"):
        for directory in (source / version / "plots", source / "plots" / version):
            if directory.is_dir():
                display(Markdown(f"#### {version}"))
                show_gallery(directory)
    table_root = source / "refined_feature_analysis_v2"
    if REGENERATE_REFINED_V2:
        try:
            if str(SCRIPTS_DIR) not in sys.path:
                sys.path.insert(0, str(SCRIPTS_DIR))
            global refined_v2
            import analyze_refined_features_v2 as refined_v2
            info = generate_refined_v2(framework)
            table_root = Path(info["output_data_dir"])
            show_gallery(Path(info["output_plot_dir"]))
        except (FileNotFoundError, ImportError) as exc:
            display(Markdown(f"V2 non régénérée : {exc}"))
    for name in ("under_invalid_only_risk.csv", "over_valid_only_risk.csv", "schema_level_context_risk.csv", "top_context_examples.csv"):
        show_csv(table_root / name)
    for target in TARGETS:
        features = MODELS_ROOT / framework / "modeling" / f"selected_{target}_features.txt"
        if features.is_file():
            values = features.read_text().splitlines()
            show_rows([{"feature retenue": x} for x in values if x], f"{target.upper()} — features retenues", limit=None)

for framework in FRAMEWORKS:
    show_refined(framework)


## 5. Meilleurs modèles GitHub : résultats enregistrés

Les scores du réentraînement final et les matrices de confusion sont affichés sur la partition **test** uniquement. Guidance UNDER n'a pas de modèle : aucun exemple positif dans les données d'entraînement correspondantes.

La validation croisée sauvegardée utilise encore d'anciennes listes de features ; elle est présentée séparément et ne doit pas être attribuée automatiquement aux modèles finaux.


In [ ]:
show_rows([r for r in read_rows(COVERAGE_ROOT / "filtered_retraining_summary.csv") if r["framework"] in FRAMEWORKS], "Configuration retenue")
show_rows([r for r in read_rows(COVERAGE_ROOT / "retained_list_retraining_scores.csv") if r["framework"] in FRAMEWORKS], "Scores finaux enregistrés")
for framework in FRAMEWORKS:
    display(Markdown(f"### {framework}"))
    for target in TARGETS:
        metrics_dir = MODELS_ROOT / framework / "metrics"
        rows = [r for r in read_rows(metrics_dir / f"{target}_metrics.csv") if r.get("split") == "test"]
        show_rows(rows, f"{target.upper()} — test enregistré")
        matrices = [r for r in read_rows(metrics_dir / f"{target}_confusion_matrix.csv") if r.get("split") == "test"]
        show_confusions(matrices, f"{framework} {target.upper()}")
    show_gallery(MODELS_ROOT / framework / "plots" / "feature_importance")
show_rows([r for r in read_rows(COVERAGE_ROOT / "cross_validation_summary.csv") if r["framework"] in FRAMEWORKS], "Validation croisée historique — vérifier le nombre de features")


### Réexécuter les modèles retenus

Les cellules suivantes chargent les pickles et exécutent `predict_proba`, sans entraînement ni modification des artefacts. Elles utilisent par défaut le Python de `.venv` pour disposer des bibliothèques des modèles, indépendamment du noyau Jupyter.

Pour XGR, `models_recovered/` est privilégié de façon cohérente. Une bibliothèque manquante ou une version scikit-learn incompatible est signalée comme erreur de calcul ; les scores enregistrés restent lisibles. Aucun score historique n'est présenté comme un recalcul réussi.


In [ ]:
MODEL_WORKER = 'import csv, json, math, sys, warnings\nfrom pathlib import Path\nimport joblib\nimport numpy as np\nfrom sklearn.exceptions import InconsistentVersionWarning\nfrom sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, roc_auc_score, average_precision_score, balanced_accuracy_score\nwarnings.filterwarnings("error", category=InconsistentVersionWarning)\nconfig = json.loads(sys.argv[1])\nroot = Path(config["coverage_root"])\nresults = []\nfor framework in config["frameworks"]:\n    for target in config["targets"]:\n        result = {"framework": framework, "target": target, "mode": config["mode"]}\n        if framework == "guidance" and target == "under":\n            result.update(status="skipped", reason="Aucun modèle Guidance UNDER : absence de cas positifs à l\'apprentissage.")\n            results.append(result)\n            continue\n        try:\n            framework_root = root / "modeles_predictifs" / framework\n            # Use the retained recovered model consistently for XGR, rather than mixing versions.\n            recovered = framework_root / "models_recovered" / f"{target}_model.pkl"\n            path = recovered if recovered.is_file() and recovered.stat().st_size else framework_root / "models" / f"{target}_model.pkl"\n            result["model_path"] = str(path)\n            model = joblib.load(path)\n            features = model["selected_features"]\n            result.update(model=model["model_name"], threshold=float(model["threshold"]), features=len(features))\n            data_path = (framework_root / "modeling" / f"{target}_dataset.csv" if config["mode"] == "test"\n                         else root / "external_eval" / config["dataset"] / framework / f"{config[\'dataset\']}_external_features.csv")\n            with data_path.open(encoding="utf-8-sig", newline="") as handle:\n                rows = list(csv.DictReader(handle))\n            if config["mode"] == "test":\n                ids = set(map(str, model["split_schema_ids"]["test"]))\n                rows = [r for r in rows if r["schema_id"] in ids]\n            else:\n                negative = "CORRECT_INVALID" if target == "under" else "CORRECT_VALID"\n                rows = [r for r in rows if r.get("actual_result") in {"passed", "failed"}\n                        and r.get("failure_type") in {target.upper(), negative}]\n            if not rows:\n                result.update(status="skipped", reason="Aucun test disponible pour ce périmètre.")\n                results.append(result)\n                continue\n            missing = sorted(set(features) - set(rows[0]))\n            if missing:\n                raise ValueError(f"Features absentes du CSV : {missing}")\n            # The fitted preprocessor stores exact categorical column indices.\n            preprocessor = model["pipeline"].steps[0][1]\n            categorical = set()\n            for name, transformer, columns in preprocessor.transformers:\n                if name == "categorical":\n                    categorical.update(columns)\n            def numeric(value):\n                text = str(value).strip().lower()\n                if text == "true": return 1.0\n                if text == "false" or not text: return 0.0\n                try: number = float(text)\n                except ValueError: return 0.0\n                return number if math.isfinite(number) else 0.0\n            matrix = np.asarray([[str(row.get(f, "")) or "absent" if i in categorical else numeric(row.get(f, 0))\n                                  for i, f in enumerate(features)] for row in rows], dtype=object)\n            y = np.asarray([int(r[f"y_{target}"]) if config["mode"] == "test" else int(r["failure_type"] == target.upper()) for r in rows])\n            probabilities = model["pipeline"].predict_proba(matrix)[:, 1]\n            predicted = (probabilities >= model["threshold"]).astype(int)\n            tn, fp, fn, tp = confusion_matrix(y, predicted, labels=[0, 1]).ravel()\n            both_classes = len(set(y)) == 2\n            result.update(status="ok", n=len(rows), positives=int(y.sum()),\n                          accuracy=float(accuracy_score(y, predicted)), precision=float(precision_score(y, predicted, zero_division=0)),\n                          recall=float(recall_score(y, predicted, zero_division=0)), f1=float(f1_score(y, predicted, zero_division=0)),\n                          pr_auc=float(average_precision_score(y, probabilities)) if both_classes else None,\n                          roc_auc=float(roc_auc_score(y, probabilities)) if both_classes else None,\n                          balanced_accuracy=float(balanced_accuracy_score(y, predicted)) if both_classes else None,\n                          tn=int(tn), fp=int(fp), fn=int(fn), tp=int(tp))\n            if not both_classes:\n                result["note"] = "Une seule classe observée : AUC non définies ; la détection des positifs ne peut pas être évaluée sans positifs."\n        except Exception as exc:\n            result.update(status="error", reason=f"{type(exc).__name__}: {exc}")\n        results.append(result)\nprint(json.dumps(results, allow_nan=False))'

PREDICTION_RESULTS = {}
def run_predictions(mode):
    config = {"coverage_root": str(COVERAGE_ROOT), "frameworks": FRAMEWORKS, "targets": TARGETS,
              "dataset": EXTERNAL_DATASET, "mode": mode}
    result = subprocess.run([MODEL_PYTHON, "-c", MODEL_WORKER, json.dumps(config)],
                            cwd=REPO_ROOT, capture_output=True, text=True, timeout=300)
    if result.returncode:
        raise RuntimeError(result.stderr[-4000:] or result.stdout[-4000:])
    rows = json.loads(result.stdout)
    PREDICTION_RESULTS[mode] = rows
    score_columns = ["framework", "target", "status", "model", "threshold", "features", "n", "positives",
                     "accuracy", "precision", "recall", "f1", "pr_auc", "roc_auc", "balanced_accuracy", "reason", "note"]
    show_rows([{k: row[k] for k in score_columns if k in row} for row in rows], f"Prédictions recalculées — {mode}", limit=None)
    for row in rows:
        if row["status"] == "ok":
            display(Markdown(f"Modèle chargé : `{row['model_path']}`"))
            show_confusions([row], f"{row['framework']} {row['target'].upper()} — {mode}")
    return rows

if RUN_MODEL_PREDICTIONS:
    recalculated_test = run_predictions("test")
else:
    display(Markdown("Recalcul désactivé ; scores enregistrés affichés ci-dessus."))


## 6. Arbres de décision et règles déduites

Pour chaque framework et cible : arbre graphique, scores de l'arbre explicatif et **liste complète des règles positives**, avec le rapport rédigé. Ces modèles explicatifs ne sont pas les Random Forest/boosting évalués dans la section précédente. Guidance UNDER n'a pas d'arbre exploitable.


In [ ]:
show_csv(COVERAGE_ROOT / "rules" / "decision_tree_rule_summary.csv", limit=None)
for framework in FRAMEWORKS:
    for target in TARGETS:
        directory = COVERAGE_ROOT / "rules" / framework / target
        display(Markdown(f"### Règles — {framework} {target.upper()}"))
        if not directory.is_dir():
            display(Markdown("Aucun arbre enregistré pour cette cible."))
            continue
        show_gallery(directory, recursive=False)
        show_csv(directory / f"{target}_selected_metrics.csv", limit=None)
        show_csv(directory / f"{target}_positive_rules.csv", limit=None)
        show_report(directory / f"{target}_rules.md")


## 7. Évaluation externe — GitHub vers Kubernetes

Les modèles GitHub sont appliqués aux features Kubernetes **sans réentraînement**. Les résultats historiques ci-dessous proviennent du run sauvegardé. Le recalcul optionnel réutilise les features existantes et les modèles retenus actuels.

**XGR OVER :** le run historique utilise le seuil 0,58 dans `models/`, tandis que le modèle récupéré retenu utilise 0,44. Un écart avec le recalcul est donc attendu et ne doit pas être masqué. Sur Kubernetes, l'absence de positifs UNDER empêche d'évaluer leur détection ; les faux positifs restent observables.


In [ ]:
external_root = COVERAGE_ROOT / "external_eval" / EXTERNAL_DATASET
historical_external = [r for r in read_rows(external_root / "external_eval_summary.csv") if r["framework"] in FRAMEWORKS and r["target"] in TARGETS]
show_rows(historical_external, "Évaluation externe enregistrée", limit=None)
for row in historical_external:
    show_confusions([row], f"{row['framework']} {row['target'].upper()} — externe enregistré")
for framework in FRAMEWORKS:
    for target in TARGETS:
        path = external_root / framework / f"{target}_external_misclassified_tests.csv"
        if path.exists():
            display(Markdown(f"#### Exemples d'erreurs externes — {framework} {target.upper()}"))
            show_csv(path, limit=5)
if RUN_EXTERNAL_PREDICTIONS:
    recalculated_external = run_predictions("external")


## 8. Expérience distincte : entraînement sur Kubernetes

Ces modèles sont appris sur une partie des schémas Kubernetes et testés sur une autre partie du même dataset. Ce ne sont pas les modèles GitHub de l'évaluation externe. Les tables de validation permettent de retrouver le candidat retenu ; seuls ses scores de test et sa matrice sont affichés. Aucun entraînement n'est lancé ici.


In [ ]:
for framework in FRAMEWORKS:
    directory = COVERAGE_ROOT / "kubernetes_eval" / framework
    display(Markdown(f"### Entraînement Kubernetes — {framework}"))
    for target in TARGETS:
        candidates = read_rows(directory / "metrics" / f"{target}_model_selection.csv")
        if not candidates:
            display(Markdown(f"{target.upper()} : aucun candidat enregistré (UNDER n'a pas de positifs ici)."))
            continue
        best = max(candidates, key=lambda r: tuple(float(r[k]) for k in ("pr_auc", "f1", "recall", "precision")))
        show_rows([best], f"{target.upper()} — sélection sur validation")
        metrics = [r for r in read_rows(directory / "metrics" / f"{target}_metrics.csv") if r.get("split") == "test" and r.get("model") == best["model"]]
        show_rows(metrics, "Test Kubernetes — modèle sélectionné")
        matrices = [r for r in read_rows(directory / "metrics" / f"{target}_confusion_matrix.csv") if r.get("split") == "test" and r.get("model") == best["model"]]
        show_confusions(matrices, f"{framework} {target.upper()} — apprentissage Kubernetes")


## Lecture des résultats

- Les données manquantes sont signalées ; un dossier absent n'est pas un score nul.
- Les sorties EDA et raffinées correspondent au dataset choisi ; les modèles GitHub sont évalués sur leurs propres schémas de test sauvegardés.
- Les scores recalculés et les scores historiques sont séparés. Les matrices ont la vérité en lignes et les prédictions en colonnes ; la classe positive signifie une erreur UNDER ou OVER.
- Les règles montrent des associations interprétables, pas des preuves de causalité.
- Pour conserver un notebook léger, enregistrer sans les sorties volumineuses des galeries après consultation.
